Please note, this is an example script to analyse one membrane replicate simulation. If your membrane system has multiple replicates, you need to adapt this code to input all the replicates, and then get the average values. 

This code calculates membrane properties including membrane thickness, area per lipid, cholesterol tilt angle, and cholesterol flip flop events (not rates) using the Lipyds python package, and saves the data to .pkl files, and then this script uses those .pkl files for analysis. For flip flop rate, please use the output of this script for the calculation.

For Lipyds installation instructions, please see Lily Wang's github https://github.com/lilyminium/2023-10-10_lipyd-example

# Identify leaflets

In [ ]:
import MDAnalysis as mda
import numpy as np
import sys
import xdrlib
import pickle

In [ ]:
u = mda.Universe(
    "PATH/TO/YOUR/TPR/FILE.tpr",  # topology
    "PATH/TO/YOUR/XTC/FILE.xtc",  # trajectory
)

In [ ]:
n_frames = len(u.trajectory)
n_atoms = len(u.atoms)

print(f"This trajectory has {n_frames} frames and {n_atoms} atoms")

In [ ]:
membrane_residues = u.select_atoms("not resname ION W WF") #To exclude water and ions
n_membrane_residues = membrane_residues.n_residues
print(n_membrane_residues)

In [ ]:
#Select headgroups that used for leaflet finder to find two leaflet
headgroups = membrane_residues.select_atoms(
    "name PO4 GL1 GL2 AM1 AM2"
)
assert "CHOL" not in headgroups.resnames  #Check if cholesterol in the headgroup selection
assert "CHYO" not in headgroups.resnames  #Check if cholesterol ester in the headgroup selection

In [ ]:
n_chol = membrane_residues.select_atoms("resname CHOL CHYO").n_residues
assert n_chol + len(headgroups.residues) == n_membrane_residues 

In [ ]:
from lipyds.leafletfinder.leafletfinder import LeafletFinder

In [ ]:
finder = LeafletFinder(
    universe=membrane_residues,
    select="name PO4 GL1 GL2 AM1 AM2",
    cutoff=30, # increase for higher accuracy but slower time
    pbc=True,
    method="spectralclustering",  # works for most bilayers
    n_leaflets=2,
    normal_axis="z",
    update_TopologyAttr=True
)

# Membrane thickness

`MembraneThickness` subclasses `GriddedBilayerAnalysisBase`, so it has a few extra attributes. It calculates values over a 2D grid.

Primarily the extra arguments are:

* `grid_bounds`: this helps define the grid. In an NPT simulation the box can change, so to define a static grid for the analysis we need to figure out how to define the overall box. The accepted values are "max", "min", "mean", and the default is "max".
* `axes`: how to define the 2D grid (which axes to use). Default is ("x", "y").
* `bin_size`: how much area to cover with each cell. Default is 2 Å (so each cell covers a 2Å x 2Å area). Values too low can lead to noise in the average values.

In [ ]:
from lipyds.analysis.thickness import MembraneThickness

In [ ]:
thickness = MembraneThickness(
    universe=membrane_residues,
    select="name PO4",   #Use phosphste group to calculate thickness
    leafletfinder=finder,
    bin_size=4,  # how much area to cover with each cell, in this case, each box covers 4x4 Å
)

thickness.run(start=None, stop=None, step=None, verbose=True) #Choose the start and stop frame, and skip frames by step.

Save results in .pkl files so that we can access them whenever we want and don't need to run the process again

In [ ]:
with open ("THICKNESS.pkl", "wb") as f:
    pickle.dump(thickness.results,f)

In [ ]:
with open ("THICKNESS.pkl", "rb") as f:
    membrane_thickness=pickle.load(f)

mean_thickness_value=np.mean(membrane_thickness.thickness_mean)
std_thickness_value=np.std(membrane_thickness.thickness_mean)
print("the membrane thickness is " + str(mean_thickness_value) + " and standard deviation is ± " + str(std_thickness_value))

#Please be aware, this is one membrane replicate mean value. 
#If we have multiple replicate and want to get the mean value of the those replicates, we can use, for example np.vstack() or other function to combine all the results and then get the average value and standard deviation
#This is same for all other properties analysis in this script

# Area per lipid 

In this algorithm, we select all headgroup points within `cutoff` around the headgroup of the residue we are looking at. The point normal of the central residue is used to define the plane to project points onto. The points are projected and a 2D Voronoi tessellation is computed (using scipy). The area of the polygon that the central residue is located within, is computed as the area per lipid.

**One area this may differ from others** is the use of the central residue point normal to define the plane. This can be a bit noisy and it may be much more robust to average the normals of all neighbours instead, as is done in the LipidTilt below.

**One idea that sounds good but doesn't work so well** is taking advantage of the triangular mesh. It *seems* like you should be able to take the center of all faces using the central residue vertex to create a new container polygon and compute the area of that. However, if I recall correctly, the APL just wound up having very extreme values. 

In [ ]:
from lipyds import AreaPerLipid

In [ ]:
apl_resnames = AreaPerLipid(
    universe=membrane_residues,
    select="name PO4 ROH ES",  # This will select every lipid in my membrane including cholesterol and cholesterol ester
    leafletfinder=finder,
    group_by_attr="resnames",
    update_leaflet_step=1,  # update every step -- increase if flip flop doesnt matter
    cutoff=30,  # cutoff to look for neighbors -- increase for accuracy but slower
)

apl_resnames.run(start=None, stop=None, step=None, verbose=True)

In [ ]:
df_resnames = apl_resnames.results_as_dataframe() #convert resluts into dataframe
with open ("RESNAME_APL_df.pkl", "wb") as f:
    pickle.dump(df_resnames,f)

In [ ]:
with open ("RESNAME_APL_df.pkl", "rb") as f:
    resname_APL=pickle.load(f)

resname_APL_noCHOL= resname_APL[(resname_APL["Label"] != "CHYO") & (resname_APL["Label"] != "CHOL")] # exclude chyo and chol from the dataframe
lipid_APL = resname_APL_noCHOL[ (resname_APL_noCHOL['Value'] <= 200) ]   #Lipyds will create a few vary large APL values for lipids, so we need to exclude those value. In this case, any value bigger than 200 Å^2 were excluded  
lipid_APL.head()

In [ ]:
membrane_mean_apl = lipid_APL[
    ["Leaflet", "Label", "Value", "Time"]
].groupby(["Leaflet", "Label"]).mean().reset_index()
membrane_mean_apl.head()

membrane_mean_sd = lipid_APL[
    ["Leaflet", "Label", "Value", "Time"]
].groupby(["Leaflet", "Label"]).std().reset_index()

In [ ]:
# These are for upper/extracellular and lower/intracellular leaflet
import math
upper = lipid_APL[ (lipid_APL['Leaflet'] == 1) ]    #this is for upper leaflet
lower = lipid_APL[ (lipid_APL['Leaflet'] == 2) ]    #this is for lower leaflet

upper_mean = upper[["Value"]].mean()
upper_sd = upper[["Value"]].std()
upper_se  = upper_sd / math.sqrt(len(upper))
print(f"Membrane upper APL is {upper_mean.values[0]:.2f} ± standard deviation {upper_sd.values[0]:.2f} or ± standard error of mean {upper_se.values[0]:.2f}")


lower_mean = lower[["Value"]].mean()
lower_sd = lower[["Value"]].std()
lower_se =  lower_sd / math.sqrt(len(lower))
print(f"Membrane lower APL is {lower_mean.values[0]:.2f} ± standard deviation {lower_sd.values[0]:.2f} or ± standard error of mean {lower_se.values[0]:.2f}")

# this is to calcluate the whole membrane APL
whole_apl = lipid_APL["Value"].mean()
whole_sd = lipid_APL["Value"].std()
whole_se = whole_sd/math.sqrt(len(lipid_APL["Value"]))
print(f"Membrane whol APL is {whole_apl:.2f} ± standard deviation {whole_sd:.2f} or ± standard error of mean {whole_se:.2f}")

In [ ]:
# devide them into headgroups

PC_upper = lipid_APL[(lipid_APL['Leaflet'] == 1) & (lipid_APL['Label'].str.endswith("PC"))] #In upper leaflet, select PC lipids
PC_lower = lipid_APL[(lipid_APL['Leaflet'] == 2) & (lipid_APL['Label'].str.endswith("PC"))] #In lower leaflet, select PC lipids
print("PC upper:" + str(PC_upper['Value'].mean()) + " ± " + str(PC_upper['Value'].std()))
print("PC lower:" + str(PC_lower['Value'].mean()) + " ± " + str(PC_lower['Value'].std()))


PE_upper = lipid_APL[(lipid_APL['Leaflet'] == 1) & (lipid_APL['Label'].str.endswith("PE"))]
PE_lower = lipid_APL[(lipid_APL['Leaflet'] == 2) & (lipid_APL['Label'].str.endswith("PE"))]
print("PE upper:" + str(PE_upper['Value'].mean()) + " ± " + str(PE_upper['Value'].std()))
print("PE lower:" + str(PE_lower['Value'].mean()) + " ± " + str(PE_lower['Value'].std()))


SM_upper = lipid_APL[(lipid_APL['Leaflet'] == 1) & (lipid_APL['Label'].str.endswith("SM"))]
SM_lower = lipid_APL[(lipid_APL['Leaflet'] == 2) & (lipid_APL['Label'].str.endswith("SM"))]
print("SM upper:" + str(SM_upper['Value'].mean()) + " ± " + str(SM_upper['Value'].std()))
print("SM lower:" + str(SM_lower['Value'].mean()) + " ± " + str(SM_lower['Value'].std()))

PS_upper = lipid_APL[(lipid_APL['Leaflet'] == 1) & (lipid_APL['Label'].str.endswith("PS"))]
PS_lower = lipid_APL[(lipid_APL['Leaflet'] == 2) & (lipid_APL['Label'].str.endswith("PS"))]
PI_lower = lipid_APL[(lipid_APL['Leaflet'] == 2) & (lipid_APL['Label'].str.endswith("PI"))]
print("PS upper:" + str(PS_upper['Value'].mean()) + " ± " + str(PS_lower['Value'].std()))
print("PS lower:" + str(PS_lower['Value'].mean()) + " ± " + str(PS_lower['Value'].std()))
print("PI lower:" + str(PI_lower['Value'].mean()) + " ± " + str(PI_lower['Value'].std()))

In [ ]:
# this is calculate the overall apl of each resname
resname_APL =  lipid_APL.groupby('Label')['Value']
resname_mean_std = resname_APL.agg(['mean', 'std']).reset_index()
resname_mean_std.columns = ['Lipid', 'Average Value', 'STD Value']


# this is calculate the apl of each resname of each leaflet
LF_resname_APL =  lipid_APL.groupby(['Leaflet','Label'])['Value']
LF_resname_mean_std = LF_resname_APL.agg(['mean', 'std']).reset_index()
LF_resname_mean_std.columns = ['Leaflet_PSout','Lipid', 'Average Value', 'STD Value']

resname_mean_std.to_excel('resname_mean_std.xlsx', index=False)
LF_resname_mean_std.to_excel('LF_resname_mean_std.xlsx', index=False)


# Tilt angle

This calculates the angle between the `select` beads and `select_end` beads to the specified `normal`. If `bilayer`, the bilayer normal is used. Unlike some other analyses, it is not the single residue point normal that is used as the normal vector. Instead, the point normals of all neighbours within `cutoff` A are averaged together to create the reference angle. I believe this to be more robust than just using a single point value. 

In [ ]:
from lipyds.analysis.tilt import LipidTilt

In [ ]:
u.select_atoms("resname CHOL").residues[0].atoms.names

In [ ]:
chol_tilt = u.select_atoms("resname CHOL")
heads_tilt = "name R1"
tails_tilt = "name R5"

In [ ]:
tilt = LipidTilt(
    universe=chol_tilt,
    leafletfinder=finder,
    select=heads_tilt,
    select_end=tails_tilt,
    cutoff=10,
    normal="bilayer",  # or -- "x", "y", "z"
)

tilt.run(start=None, stop=None, step=None, verbose=True)

In [ ]:
import pickle
with open ("Tilt_results.pkl", "wb") as f:
          pickle.dump(tilt.results, f)

In [ ]:
df_tilt = tilt.results_as_dataframe()
mean_values = df_tilt.groupby("Leaflet")["Value"].mean()
std_values = df_tilt.groupby("Leaflet")["Value"].std()


with open ("Tilt_result_df.pkl", "wb") as f:
          pickle.dump(df_tilt, f)


In [ ]:
with open ("Tilt_result_df.pkl", "rb") as f:
    Tilt_result=pickle.load(f)

In [ ]:
df_tilts=Tilt_result[Tilt_result.Property == "tilts"]
mean_values = df_tilts.groupby("Leaflet")["Value"].mean()
std_values = df_tilts.groupby("Leaflet")["Value"].std()
print(mean_values)
print(std_values)

# Flip flop

Out of all analyses here this is the least developed -- e.g. it does not have a `summary_as_dataframe` method. Instead it just outputs a dictionary of events, or you can check the raw data yourself. 
 
For each cholesterol we select all neighbours within `cutoff` radius, in each leaflet. We calculate the distance between the cholesterol bead and each of these neighbours. We select the minimum distance. If this minimum distance is less than `leaflet_width`, the cholesterol is considered to be part of the leaflet containing the neighbour.
 
A molecule can be in the upper leaflet, lower leaflet, or interstitial space. A translocation event only covers events from leaflet to leaflet. 

In [ ]:
from lipyds.analysis.flipflop import LipidFlipFlop

In [ ]:
chol_ff = u.select_atoms("resname CHOL")
print(chol_ff.residues[0].atoms.names)

In [ ]:
chol_flipflop = LipidFlipFlop(
    universe=chol_ff,
    leafletfinder=finder,
    select="name R2", # to pick something closer to the middle
    leaflet_width=11,  # within this distance, a bead is considered to be part of that leaflet
    cutoff=5, # cutoff radius (A) to select neighbours in leaflets to calculate distances
)
chol_flipflop.run(start=None, stop=None, step=None, verbose=True)

In [ ]:
with open ("flipflop_results.pkl", "wb") as f:
          pickle.dump(chol_flipflop.results, f)

In [ ]:
with open ("flipflop_results.pkl", "rb") as f:
    flipflop_value=pickle.load(f)

print(flipflop_value.flips_by_attr)
print(flipflop_value.flops_by_attr)
#Please be aware, this calculates how many flip and flop events happened in the analysed trajectory. To calculate the rate, we need to consider cholester number and time length of analysed trajectory.